# 02 · Baseline inference (TRAIN held-out)
> **Status:** created locally, syntax-checked only. **NOT yet run on Colab** — update docs/PROGRESS.md after the first real run.

Order: gold transcript 5 → gold transcript 40 → ASR 40. Results resume per sample in Drive.

In [ ]:
# Bootstrap: mount Drive, clone/update repo (token only from Colab Secrets, never hardcoded)
REPO_URL = "https://github.com/Quang1104/lifemate_superklang.git"
BRANCH = "claude/zalo-lifemate-handoff-5c2f7d"
REPO = "/content/lifemate_superklang"
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
url = REPO_URL
try:
    from google.colab import userdata
    url = REPO_URL.replace("https://", "https://x-access-token:" + userdata.get("GITHUB_TOKEN") + "@")
except Exception:
    pass  # public repo, or secret GITHUB_TOKEN not set
if os.path.isdir(REPO + "/.git"):
    subprocess.run(["git", "-C", REPO, "pull", "-q", "--ff-only", url, BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
from src.bench import colab_setup as cs, evaluate
cfg = cs.load_config()
print(cs.gpu_report())
DATA_DIR = cs.restore_dataset(cfg)          # unzip Drive archive -> runtime disk (skipped if present)
RESULTS = cs.results_dir(cfg)                # Drive: /MyDrive/lifemate/results
GOLD = evaluate.ensure_gold(DATA_DIR, RESULTS / "gold_validation.jsonl")  # TRAIN held-out hashes, CPU
print(DATA_DIR, GOLD)

In [ ]:
PLANNER_KEY = "planner"   # or "planner_alt" (Qwen3-4B-Instruct-2507)
ADAPTER = None            # e.g. RESULTS.parent / "checkpoints/lora_v1/adapter"
USE_ASR = True            # loads Whisper now so later cells can use it
SAMPLE_SECONDS = 30.0     # official limit; untimed score is reported too
from src.agent.local_model import LocalPlanner, LocalWhisper, initialize_runtime
from src.agent.runner import Limits
initialize_runtime()
planner = LocalPlanner(cs.fetch_model(cfg, PLANNER_KEY), device="cuda", adapter_dir=ADAPTER)
asr = LocalWhisper(cs.fetch_model(cfg, "asr"), device="cuda") if USE_ASR else None
total = planner.parameter_count + (asr.parameter_count if asr else 0)
print(f"LLM {planner.parameter_count:,}  total {total:,}")
assert planner.parameter_count < 5_000_000_000 and total < 6_000_000_000, "LifeMate parameter limits"

In [ ]:
limits = Limits(sample_seconds=SAMPLE_SECONDS, planner_seconds=SAMPLE_SECONDS)
s = evaluate.run(DATA_DIR, evaluate.select_ids(limit=5), GOLD, planner, RESULTS / f"{PLANNER_KEY}_gold_smoke5", limits=limits)
s

Continue only if the 5-sample smoke produced valid predictions (no `exception` statuses).

In [ ]:
s = evaluate.run(DATA_DIR, evaluate.select_ids(limit=40), GOLD, planner, RESULTS / f"{PLANNER_KEY}_gold_smoke40", limits=limits)
s

In [ ]:
s = evaluate.run(DATA_DIR, evaluate.select_ids(limit=40), GOLD, planner, RESULTS / f"{PLANNER_KEY}_asr_smoke40", transcribe=asr, limits=limits)
s